# 03 — MLP Surrogate
**Model:** Multi-Layer Perceptron (Neural Network) — numpy only, no torch/tensorflow.

**Why MLP here:**
MLP can fit non-linear, non-monotonic trends (important given the non-trivial
C11/C12 variation you see at mid-Cr compositions). With only 17 points it is
prone to overfitting, so we use strong regularisation (L2 weight decay) and LOO-CV.

**Architecture:**
```
Input(1) → Dense(16, tanh) → Dense(16, tanh) → Dense(1, linear)
```
Small on purpose — more neurons would overfit badly at N=17.

**Training:** Mini-batch gradient descent with Adam optimiser (numpy implementation).

**Limitation:** No native uncertainty output. Bootstrap ensemble used to get
approximate prediction intervals.

**Outputs saved to** `../analysis/`:
- `mlp_predictions.png`
- `mlp_loo_residuals.png`
- `mlp_results.pkl`


---
## Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('data_utils.py')))
from data_utils import load_data, plot_loo_residuals, TARGETS, PALETTE
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size':12,'figure.dpi':120})
print('Imports OK — numpy-only MLP, no torch/tensorflow')

---
## Cell 2 — Load Data

In [ ]:
DATA_PATH = '../analysis/elastic_constants_fecr.csv'
d = load_data(DATA_PATH)
df          = d['df']
X           = d['X']
X_pred      = d['X_pred']
x_pred_atoms= d['x_pred_atoms']
targets     = d['targets']
noise_gpa   = d['noise_gpa']   # σ per point — three tiers
flagged     = d['flagged']     # bool: Tier C
tier        = d['tier']        # 'A', 'B', 'C'

# ── Sample weights: inverse noise variance, normalised ───────────────────────
# Tier A (conv_thr=1e-8, FM):  σ=1.0 GPa  → weight=high
# Tier B (conv_thr=1e-7, FM):  σ=2.0 GPa  → weight=medium
# Tier C (conv_thr=1e-5, AFM): σ=10.0 GPa → weight=low
w = 1.0 / (noise_gpa ** 2)      # inverse variance
w = w / w.sum() * len(w)         # normalise: mean weight = 1

print('\nSample weights by tier:')
for t in ['A','B','C']:
    m = tier == t
    if m.any():
        print(f'  Tier {t}: σ={noise_gpa[m][0]:.1f} GPa  '
              f'→  weight={w[m][0]:.4f}  ({m.sum()} tags)')


---
## Cell 3 — MLP Implementation (numpy)

### What each component does:

**Weights and biases:** Randomly initialised (He init — scaled by √(2/n_in)).
Why He? Prevents vanishing/exploding activations in tanh networks at init time.

**Forward pass:** Input → linear transform + bias → tanh activation → repeat → output.

**Loss:** Weighted MSE + L2 regularisation on all weights.
```
L = Σ w_i·(y_i − ŷ_i)² + λ·Σ ||W||²
```

**Backward pass:** Chain rule through each layer (backpropagation).

**Adam optimiser:** Adaptive per-parameter learning rates using first and second
moment estimates. More robust than plain SGD for small noisy datasets.
```
m = β₁·m + (1−β₁)·g        # first moment (mean of gradients)
v = β₂·v + (1−β₂)·g²       # second moment (variance of gradients)
θ = θ − α · m̂ / (√v̂ + ε)  # parameter update
```

In [ ]:
class MLP:
    """
    Two-hidden-layer MLP for scalar regression.
    Architecture: 1 → h1 → h2 → 1  (tanh activations, linear output)
    Trained with Adam + L2 weight decay + sample weights.

    Parameters
    ----------
    hidden  : tuple  — neurons per hidden layer, e.g. (16, 16)
    lr      : float  — Adam learning rate
    n_epochs: int    — training epochs
    lam     : float  — L2 weight decay
    seed    : int
    """
    def __init__(self, hidden=(16,16), lr=1e-2, n_epochs=3000, lam=1e-3, seed=42):
        self.hidden   = hidden
        self.lr       = lr
        self.n_epochs = n_epochs
        self.lam      = lam
        self.seed     = seed
        self.loss_history = []

    def _init_params(self, layer_sizes):
        """He initialisation for all weight matrices."""
        rng = np.random.default_rng(self.seed)
        self.W, self.b = [], []
        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.W.append(rng.normal(0, np.sqrt(2./n_in), (n_in, n_out)))
            self.b.append(np.zeros(n_out))

    def _forward(self, X):
        """Forward pass. Returns activations at each layer."""
        A = [X]          # A[0] = input
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            Z = A[-1] @ W + b
            if i < len(self.W) - 1:  # hidden layers
                A.append(np.tanh(Z))
            else:                     # output layer: linear
                A.append(Z)
        return A

    def _backward(self, A, y, sw):
        """
        Backpropagation.
        Returns list of (dW, db) gradients, one per layer.
        sw: sample weights (n,)
        """
        n   = len(y)
        dWs, dbs = [], []
        # Output layer delta: weighted MSE gradient
        pred = A[-1].flatten()
        delta = -2 * sw * (y - pred) / n    # (n,)
        delta = delta.reshape(-1, 1)
        for i in reversed(range(len(self.W))):
            dW = A[i].T @ delta + self.lam * self.W[i]
            db = delta.sum(axis=0)
            dWs.insert(0, dW)
            dbs.insert(0, db)
            if i > 0:   # backprop through tanh
                delta = (delta @ self.W[i].T) * (1 - A[i]**2)
        return dWs, dbs

    def fit(self, X, y, sample_weight=None):
        """Train MLP with Adam optimiser."""
        if sample_weight is None:
            sample_weight = np.ones(len(y))
        sw = sample_weight / sample_weight.sum() * len(y)

        # Normalise inputs and targets for stable training
        self.X_mean, self.X_std = X.mean(), X.std() + 1e-8
        self.y_mean, self.y_std = y.mean(), y.std() + 1e-8
        Xn = (X - self.X_mean) / self.X_std
        yn = (y - self.y_mean) / self.y_std

        sizes = [1] + list(self.hidden) + [1]
        self._init_params(sizes)

        # Adam state
        m = [np.zeros_like(W) for W in self.W]
        v = [np.zeros_like(W) for W in self.W]
        mb = [np.zeros_like(b) for b in self.b]
        vb = [np.zeros_like(b) for b in self.b]
        beta1, beta2, eps_adam = 0.9, 0.999, 1e-8

        self.loss_history = []
        for epoch in range(1, self.n_epochs + 1):
            A    = self._forward(Xn)
            pred = A[-1].flatten()
            loss = np.sum(sw * (yn - pred)**2) / len(yn) + \
                   self.lam * sum(np.sum(W**2) for W in self.W)
            self.loss_history.append(loss)
            dWs, dbs = self._backward(A, yn, sw)
            for i in range(len(self.W)):
                m[i]  = beta1*m[i]  + (1-beta1)*dWs[i]
                v[i]  = beta2*v[i]  + (1-beta2)*dWs[i]**2
                mb[i] = beta1*mb[i] + (1-beta1)*dbs[i]
                vb[i] = beta2*vb[i] + (1-beta2)*dbs[i]**2
                mc = m[i]  / (1 - beta1**epoch)
                vc = v[i]  / (1 - beta2**epoch)
                mbc= mb[i] / (1 - beta1**epoch)
                vbc= vb[i] / (1 - beta2**epoch)
                self.W[i] -= self.lr * mc  / (np.sqrt(vc)  + eps_adam)
                self.b[i] -= self.lr * mbc / (np.sqrt(vbc) + eps_adam)
        return self

    def predict(self, X):
        Xn = (X - self.X_mean) / self.X_std
        A  = self._forward(Xn)
        return A[-1].flatten() * self.y_std + self.y_mean

print('MLP class defined')

---
## Cell 4 — Bootstrap Uncertainty

Since MLP has no analytic uncertainty, we use **bootstrap ensembling**:
train N_BOOT models on random samples-with-replacement of the data,
then use their prediction spread as an approximate confidence interval.

With N=17, bootstrap std is a rough estimate — treat it as indicative, not calibrated.

In [ ]:
def bootstrap_predict(X_tr, y_tr, sw, X_pred, n_boot=50,
                       hidden=(16,16), lr=1e-2, n_epochs=3000, lam=1e-3, seed=0):
    """
    Returns mean and std of N_BOOT bootstrapped MLP predictions at X_pred.
    Each bootstrap model trains on a resample-with-replacement of the data.
    """
    rng  = np.random.default_rng(seed)
    preds = []
    n = len(X_tr)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        Xb, yb, wb = X_tr[idx], y_tr[idx], sw[idx]
        m = MLP(hidden=hidden, lr=lr, n_epochs=n_epochs, lam=lam, seed=seed+b)
        m.fit(Xb, yb, sample_weight=wb)
        preds.append(m.predict(X_pred))
    preds = np.array(preds)   # (n_boot, n_pred)
    return preds.mean(axis=0), preds.std(axis=0)

print('Bootstrap function defined')

---
## Cell 5 — Hyperparameter Search + LOO-CV

In [ ]:
# Hyperparameter grid
HIDDEN_OPTS = [(8,8), (16,16)]
LR_OPTS     = [5e-3, 1e-2]
LAM_OPTS    = [1e-3, 1e-2]
N_EPOCHS    = 3000

def mlp_loo(X, y, sw, hidden, lr, lam, n_epochs):
    loo = LeaveOneOut()
    preds, trues = [], []
    for tr, te in loo.split(X):
        m = MLP(hidden=hidden, lr=lr, n_epochs=n_epochs, lam=lam)
        m.fit(X[tr], y[tr], sample_weight=sw[tr])
        preds.append(float(m.predict(X[te])))
        trues.append(float(y[te[0]]))
    preds, trues = np.array(preds), np.array(trues)
    return mean_absolute_error(trues, preds), r2_score(trues, preds), preds, trues

best_mlp_params = {}
for tname in TARGETS:
    y = targets[tname]
    print(f'\n── {tname} ─────────────────────────────────────')
    best_mae = np.inf
    for hid in HIDDEN_OPTS:
        for lr in LR_OPTS:
            for lam in LAM_OPTS:
                mae, r2, _, _ = mlp_loo(X, y, w, hid, lr, lam, N_EPOCHS)
                print(f'  h={str(hid):8s} lr={lr:.0e} λ={lam:.0e}  MAE={mae:.2f} GPa  R²={r2:.4f}')
                if mae < best_mae:
                    best_mae = mae
                    best_mlp_params[tname] = {'hidden':hid,'lr':lr,'lam':lam}
    print(f'  → Best: {best_mlp_params[tname]}  (MAE={best_mae:.2f})')

---
## Cell 6 — Train Final Models + Bootstrap Uncertainty

In [ ]:
mlp_models  = {}
mlp_results = {}
N_BOOT = 50   # bootstrap ensemble size — increase for smoother bands (slower)

for tname in TARGETS:
    y  = targets[tname]
    p  = best_mlp_params[tname]
    # Final single model
    model = MLP(hidden=p['hidden'], lr=p['lr'], n_epochs=N_EPOCHS, lam=p['lam'])
    model.fit(X, y, sample_weight=w)
    mlp_models[tname] = model
    # Bootstrap uncertainty bands
    print(f'{tname}: running bootstrap (n={N_BOOT})...')
    mu, sig = bootstrap_predict(X, y, w, X_pred, n_boot=N_BOOT,
                                 hidden=p['hidden'], lr=p['lr'],
                                 n_epochs=N_EPOCHS, lam=p['lam'])
    # LOO-CV
    mae, r2, loo_preds, loo_true = mlp_loo(X, y, w, p['hidden'], p['lr'], p['lam'], N_EPOCHS)
    mlp_results[tname] = {
        'mu': mu, 'std': sig,
        'mae': mae, 'r2': r2,
        'loo_true': loo_true, 'loo_preds': loo_preds,
        'residuals': loo_true - loo_preds,
        'loss_history': model.loss_history,
        'best_params': p
    }
    print(f'  LOO MAE={mae:.2f} GPa  R²={r2:.4f}')

---
## Cell 7 — Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, tname in zip(axes, TARGETS):
    loss = mlp_results[tname]['loss_history']
    ax.semilogy(range(1, len(loss)+1), loss, color=PALETTE[tname], lw=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)')
    ax.set_title(f'{tname} — Training Loss')
    ax.grid(alpha=0.3)
plt.suptitle('MLP: Training Loss vs Epoch', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/mlp_training_loss.png', bbox_inches='tight')
plt.show(); print('Saved: mlp_training_loss.png')

---
## Cell 8 — Prediction Plots with Bootstrap Bands

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, tname in zip(axes, TARGETS):
    res = mlp_results[tname]
    mu, sig, col = res['mu'], res['std'], PALETTE[tname]
    ax.fill_between(x_pred_atoms, mu-2*sig, mu+2*sig, alpha=0.12, color=col, label='±2σ bootstrap')
    ax.fill_between(x_pred_atoms, mu-sig,   mu+sig,   alpha=0.28, color=col, label='±1σ bootstrap')
    ax.plot(x_pred_atoms, mu, '-', color=col, lw=2, label='MLP mean')
    ax.scatter(df['n_cr'][~flagged], targets[tname][~flagged],
               color='black', s=50, zorder=5, label='DFT')
    ax.scatter(df['n_cr'][flagged],  targets[tname][flagged],
               color='tomato', s=70, marker='D', zorder=5, label='⚠️ flagged')
    ax.set_xlabel('Cr atoms (out of 16)'); ax.set_ylabel(f'{tname} (GPa)')
    ax.set_title(f'{tname} — MLP'); ax.legend(fontsize=9)
    ax.grid(alpha=0.3); ax.set_xlim(-0.5, 16.5)
plt.suptitle('MLP Surrogate: Fe-Cr Elastic Constants', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/mlp_predictions.png', bbox_inches='tight')
plt.show(); print('Saved: mlp_predictions.png')

---
## Cell 9 — LOO-CV Residuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tname in zip(axes, TARGETS):
    res = mlp_results[tname]
    plot_loo_residuals(ax, res['residuals'], df['n_cr'].values, tier,
                       f"{tname} LOO-CV\nMAE={res['mae']:.2f} GPa | R²={res['r2']:.3f}")
plt.suptitle('MLP LOO-CV Residuals  (red = flagged)', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/mlp_loo_residuals.png', bbox_inches='tight')
plt.show(); print('Saved: mlp_loo_residuals.png')

---
## Cell 10 — Export Results

In [ ]:
with open('../analysis/mlp_results.pkl','wb') as f:
    pickle.dump(mlp_results, f)
print('Saved: ../analysis/mlp_results.pkl')
print('\nMLP LOO-CV Summary:')
for t,v in mlp_results.items():
    p = v['best_params']
    print(f'  {t}: h={p["hidden"]}  lr={p["lr"]:.0e}  λ={p["lam"]:.0e}  '
          f'MAE={v["mae"]:.2f} GPa  R²={v["r2"]:.4f}')